# 08 — Optimization Sanity

**Module:** `fleximorpv2/baseline_optimization.py`

Objective sign, constraints, monotonicity sweeps.

In [ ]:
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("error", category=RuntimeWarning)


def _repo_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "fleximorpv2").is_dir():
            return candidate
    raise RuntimeError(
        "Could not find fleximorp-project root. "
        "Open Jupyter from the repo or a notebooks/ subdirectory."
    )


_repo = _repo_root()
_audit = _repo / "notebooks" / "audit"
for path in (_repo, _audit):
    path_str = str(path)
    if path_str not in sys.path:
        sys.path.insert(0, path_str)

from _audit_helpers import (
    REPO_ROOT,
    OUTPUT_DIR,
    SITE_OUTPUT_DIR,
    reload_fleximorp,
    assert_close,
    assert_energy_balance,
    assert_cf_bounds,
    assert_positive,
)

reload_fleximorp()
np.random.seed(42)
print(f"Repo root: {REPO_ROOT}")
print(f"Audit outputs: {OUTPUT_DIR}")
print(f"Site outputs: {SITE_OUTPUT_DIR}")

## Checklist

- [ ] Optimal LCOE ≤ random nearby designs
- [ ] Production target enforced
- [ ] Max capacity respected
- [ ] Sweep one capacity variable — LCOE has sensible shape
- [ ] differential_evolution vs scipy within tolerance

In [ ]:
from fleximorpv2.config import load_config
from fleximorpv2.baseline_optimization import BaselineOptimization

config = load_config("alaska")
config.uncertainty["monte_carlo_runs"] = 10
opt = BaselineOptimization(config)

# TODO: run optimize with fixed seed; then compare to perturbed designs
results = opt.optimize("production", 200_000, method="scipy")
assert_positive(results.financial_metrics["lcoe"], label="baseline LCOE")
print(results.financial_metrics)
print(results.technical_metrics)